# 08 · Prueba con video de voluntarios (tarea 2.6) — DATOS REALES, EJECUCIÓN ÚNICA

Reglas: `docs/sprint2/plan_evaluacion.md`, sección "Prueba con video". El código es **el mismo** que se ensayó en `07_ensayo_video.ipynb`
(`src/video.py`, `src/video_verificacion.py`).

**Protecciones** (las mismas de la tarea 2.5, más las de privacidad):
- Se niega a ejecutarse sin la variable `FACEID_EJECUTAR_VIDEO=SI`.
- Exige el árbol de git limpio y registra el commit.
- Si ya hay resultados (`09_video_resultados.json`), exige `FACEID_MOTIVO_REEJECUCION`.
- Usa el umbral oficial **0,5514**, que no se recalcula, y los parámetros fijos de `src/config.py`.
- **Privacidad:** el video, las anotaciones y las fotos deben estar **fuera del repositorio** (o en `data/`). Los voluntarios se identifican con
  seudónimos (`V01`…). El cuaderno **no muestra ni guarda imágenes**. En `docs/` solo se guardan **métricas agregadas**; el detalle por
  fotograma (sin imágenes) va a `data/video_real/`, que git ignora.

Antes de ejecutarlo:
1. Registrar a cada voluntario desde la **app web** (Administración) y definir `FACEID_ALMACEN_APP` con la carpeta de datos de la app (la misma que `FACEID_APP_DATA`), para que el video y la demostración usen el mismo registro. Alternativa: `src.registro_voluntario`.
2. Preparar `anotaciones.csv` con las columnas `inicio_s,fin_s,identidad`. La identidad es `V01`… o `no registrado` (se admite `no registrado A`, etc.).
3. Commitear cualquier cambio y ejecutar en PowerShell:

```
$env:FACEID_EJECUTAR_VIDEO = "SI"; $env:FACEID_VIDEO = "D:/privado/video.mp4"; $env:FACEID_ANOTACIONES = "D:/privado/anotaciones.csv"
.venv/Scripts/python -m jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 notebooks/08_video_voluntarios.ipynb
```

In [ ]:
import os, sys, json, time, subprocess, platform
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config as C
SALIDA = C.DOCS / "09_video_resultados.json"
if os.environ.get("FACEID_EJECUTAR_VIDEO") != "SI":
    raise RuntimeError("Prueba con video BLOQUEADA: define FACEID_EJECUTAR_VIDEO=SI solo cuando el usuario lo autorice y existan los datos reales.")
MOTIVO = os.environ.get("FACEID_MOTIVO_REEJECUCION")
if SALIDA.exists() and not MOTIVO:
    raise RuntimeError(f"Ya existe {SALIDA.name}: la prueba con video es de ejecución única. Para re-ejecutar define FACEID_MOTIVO_REEJECUCION.")
git = lambda *a: subprocess.run(["git", *a], cwd=ROOT, capture_output=True, text=True).stdout.strip()
REGISTRO = {"commit": git("rev-parse", "HEAD"), "arbol_limpio": git("status", "--porcelain") == "",
            "inicio": time.strftime("%Y-%m-%d %H:%M:%S"), "motivo_reejecucion": MOTIVO}
assert REGISTRO["arbol_limpio"], "Hay cambios sin commitear: commitea antes de ejecutar."
print(REGISTRO)

In [ ]:
import os, sys, json, time, subprocess, platform
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from src import config as C, data as D, pipeline as P, video as VID, video_verificacion as VV
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e6e6", "axes.axisbelow": True})
pd.set_option("display.width", 200)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
PKG = C.MODELS_DIR / "verificador_facenet_vggface2"
from src import video as VID
VIDEO = VID.check_private_path(os.environ["FACEID_VIDEO"])
ANOT = VID.check_private_path(os.environ["FACEID_ANOTACIONES"])
# Fuente de las plantillas de voluntarios: el almacén de la app web (mismo registro que la demostración, solo lectura)
# si se define FACEID_ALMACEN_APP; si no, el archivo plantillas_voluntarios.npz del paquete (src.registro_voluntario).
ALMACEN = os.environ.get("FACEID_ALMACEN_APP")
ARCH = None if ALMACEN else C.PLANTILLAS_VOLUNTARIOS
if ARCH:
    assert (PKG / ARCH).exists(), "Registra primero a los voluntarios (app web o src.registro_voluntario)"
v = VV.cargar_verificador(PKG, DEV, ARCH, ALMACEN)
vol = [u for u, g in zip(v.users, v.group) if g == "voluntario"]
assert not any(u.upper().startswith("ENSAYO") for u in vol), "Hay plantillas de ENSAYO en el archivo real"
assert abs(v.threshold - C.UMBRAL_OFICIAL) < 1e-9 and v.cfg["umbral_estado"] == "propuesto en validación"
print("Voluntarios registrados (seudónimos):", vol, "| fuente:", "almacén de la app" if ALMACEN else ARCH)

In [ ]:
t0 = time.time()
info, met, sin, frames, comps = VV.run(VIDEO, ANOT, voluntarios=ARCH, device=DEV, almacen_app=ALMACEN)
info["segundos_total"] = time.time() - t0
priv = C.ROOT / "data" / "video_real"; priv.mkdir(parents=True, exist_ok=True)
frames.to_csv(priv / "detalle_fotogramas.csv", index=False)      # sin imágenes; fuera de git
met.to_csv(C.DOCS / "09_video_metricas.csv", index=False); sin.to_csv(C.DOCS / "09_video_sin_rostro.csv", index=False)
print({k: v_ for k, v_ in info.items() if k != "voluntarios"})
def graficar(met, titulo, archivo):
    """Solo metricas AGREGADAS (sin imagenes ni datos por fotograma)."""
    t = met[met.persona == "todas"].copy()
    t["etiqueta"] = t.tipo + " (" + t.metrica + ")"
    fig, ax = plt.subplots(figsize=(9, 3.8))
    niveles = ["fotograma", "ventana"]; col = {"fotograma": "#2a78d6", "ventana": "#eb6834"}
    etiquetas = list(dict.fromkeys(t.etiqueta))
    y = np.arange(len(etiquetas))
    for k, nv in enumerate(niveles):
        d = t[t.nivel == nv].set_index("etiqueta").reindex(etiquetas)
        ax.barh(y + (k - 0.5) * 0.38, d.tasa * 100, height=0.36, color=col[nv], label=nv, edgecolor="white", linewidth=2)
        for yy, (v, e, n) in zip(y + (k - 0.5) * 0.38, zip(d.tasa, d.errores, d.n)):
            if not np.isnan(v): ax.text(v * 100 + 0.3, yy, f"{v*100:.1f} % ({int(e)}/{int(n)})", va="center", fontsize=7, color="#333")
    ax.set_yticks(y); ax.set_yticklabels(etiquetas, fontsize=8); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set_xlabel("tasa de error (%)"); ax.legend(frameon=False, fontsize=8, loc="lower right"); ax.set_title(titulo, loc="left", fontsize=10)
    plt.tight_layout(); plt.savefig(C.FIGS / archivo); plt.show()

def fmt(met):
    m = met.copy(); m["tasa_%"] = (m.tasa * 100).round(2); m["IC95_%"] = m.IC95.map(lambda t: f"{t[0]*100:.1f}-{t[1]*100:.1f}")
    return m.drop(columns=["tasa", "IC95"])
fmt(met[met.persona == "todas"])

In [ ]:
graficar(met, "Prueba con video de voluntarios: tasas de error por fotograma y por ventana", "09_video_tasas.png")
fmt(met[met.persona != "todas"]).sort_values(["nivel", "tipo", "persona"])

In [ ]:
sin

## Tiempo por verificación incluyendo la detección (GPU y CPU), con fotogramas del video real que solo están en memoria

In [ ]:
ann = VID.read_annotations(ANOT)
fr = VID.sample_frames(VIDEO, 40, C.VIDEO_CADA_N, ann)
tiempos = VID.timing(fr, lambda d: VV.cargar_verificador(PKG, d, ARCH, ALMACEN), lambda d: VID.FaceCropper(d)); del fr
tiempos["cumple_3s_p95"] = tiempos.p95_s <= 3.0
tiempos.to_csv(C.DOCS / "09_video_tiempos.csv", index=False); tiempos.round(4)

In [ ]:
REGISTRO["fin"] = time.strftime("%Y-%m-%d %H:%M:%S")
json.dump({"registro": REGISTRO, "info": info, "metricas": met.to_dict("records"), "sin_rostro": sin.to_dict("records"),
           "tiempos": tiempos.to_dict("records")}, open(SALIDA, "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=str)
print("Guardado (solo agregados):", SALIDA.name)

## Conclusiones

*(Se redactan después de la ejecución única, en `docs/sprint2/`, sin modificar las celdas anteriores.)*